# KCORC Summer School - TESPy Workshop

## Case Study: 5.5 MWe Double-stage ORC Kirchstockach

The Kirchstockach geothermal power plant is a two-stage Organic Rankine Cycle
(ORC) system consisting of a High-Temperature (HT) ORC and a Low-Temperature
(LT) ORC operating in series. The geothermal brine first transfers heat to the
HT cycle and subsequently to the LT cycle before reinjection.

![ORC flowsheet](../orc.svg)

Figure 1: Scheme of the double-stage ORC power plant in Kirchstockach, Germany
(Florian Heberle, Thomas Jahrfeld and Dieter Brüggemann, 2015)
https://worldgeothermal.org/pdf/IGAstandard/WGC/2015/26002.pdf


### Objective

Your task is to design and integrate the High-Temperature Organic Rankine Cycle
(HT-ORC) using the operating data reported in the reference paper. Starting
from the provided TESPy model of the Low-Temperature ORC (LT-ORC), build the
HT-ORC and connect it to the geothermal heat source.

The objective is to estimate design-point performance of the plant and
understand how the two ORC modules interact through the geothermal heat source.

### Tasks


1. Run the given script for LT-ORC. Please, compare the results with the values
   provided in the Heberle et al. paper. Find the differences between the model
   provided here and the data from the paper.
2. Create the T-s diagram of the ORC system as well as the Q-T diagrams of the
   heat exchangers.
3. Modify the boundary conditions specified in this model to match the paper's
   reported design as close as possible.
4. Build the HT-ORC using TESPy. Extend the provided script with the HT-ORC and
   connect it to the geothermal source upstream of point C. Compare the results
   with the values provided in the Heberle et al. paper.
5. Perform two sensitivity analyses, in which you evaluate the influence of the
   respective parameter on power output of both, LT- and HT-ORC systems, and
   overall performance of the combined HT-/ LT-ORC system as well as the
   geothermal re-injection temperature. Also investigate the system's overall
   UA (sum of the UA of all heat exchangers) as a proxy for the investment
   costs.

   - Vary the geothermal split ratio.
   - Vary the geothermal fluid temperature at point C (the LT-evaporator inlet
     temperature) from 85 °C to 99 °C.

6. Integrate the model into TESPy's ModelTemplate class and perform a multi-
   variate optimization to maximize the power output and minimize UA 
   simultaneously.

## LT-ORC (Low-Temperature Organic Rankine Cycle)

This notebook builds the **LT-ORC branch only** (working fluid: R245fa),
coupled to the geothermal water and to the ambient air used to cool the
condenser.

### 1. Imports

In [ ]:
from tespy.networks import Network
from tespy.components import (
    Source, Sink,
    Pump, Turbine,
    SectionedHeatExchanger,
    CycleCloser,
    Splitter,
    PowerSink, PowerBus,
    Motor, Generator
)
from tespy.connections import Connection, PowerConnection

import numpy as np
import matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI

### 2. Network and units

In [ ]:
nw = Network()
nw.units.set_defaults(
    temperature="degC", pressure="bar", pressure_difference="bar",
    enthalpy="kJ/kg", mass_flow="kg/s", power="kW", heat="kW"
)

### 3. Components

In [ ]:
lt_cc = CycleCloser("LT-cycle-closer")

lt_pump = Pump("LT-pump")
lt_preheater = SectionedHeatExchanger("LT-preheater")
lt_evaporator = SectionedHeatExchanger("LT-evaporator")
lt_turbine = Turbine("LT-turbine")
lt_condenser = SectionedHeatExchanger("LT-condenser")

geo_source = Source("geothermal-source")                    # point C
geo_split = Splitter("geothermal-splitter", num_out=2)       # point D
geo_sink_preheater = Sink("geothermal-sink (LT-preheater)")  # point E
geo_sink_other = Sink("geothermal-sink (to HT-preheater)")

lt_air_source = Source("LT-air-source")
lt_air_sink = Sink("LT-air-sink")

### 4. Connections -> working fluid loop (R245fa)

In [ ]:
c7 = Connection(lt_condenser, "out1", lt_pump, "in1", label="7")
c8 = Connection(lt_pump, "out1", lt_preheater, "in2", label="8")
c9 = Connection(lt_preheater, "out2", lt_evaporator, "in2", label="9")
c10 = Connection(lt_evaporator, "out2", lt_turbine, "in1", label="10")
c10a = Connection(lt_turbine, "out1", lt_cc, "in1", label="10a")
c11 = Connection(lt_cc, "out1", lt_condenser, "in1", label="11")  # same physical state as c10a, enforced equal by the CycleCloser

nw.add_conns(c7, c8, c9, c10, c10a, c11)

### 5. Connections -> geothermal water side

In [ ]:
gC = Connection(geo_source, "out1", lt_evaporator, "in1", label="C")
gD = Connection(lt_evaporator, "out1", geo_split, "in1", label="D")
gD_ph = Connection(geo_split, "out1", lt_preheater, "in1", label="E")
gD_ht = Connection(geo_split, "out2", geo_sink_other, "in1", label="D2")
gE = Connection(lt_preheater, "out1", geo_sink_preheater, "in1", label="F")

nw.add_conns(gC, gD, gD_ph, gD_ht, gE)

### 6. Connections -> air side of the condenser

In [ ]:
lt_a1 = Connection(lt_air_source, "out1", lt_condenser, "in2", label="LT-a1")
lt_a2 = Connection(lt_condenser, "out2", lt_air_sink, "in1", label="LT-a2")

nw.add_conns(lt_a1, lt_a2)

### 7. PowerConnections -> generator and motor

In [ ]:
lt_generator = Generator("LT-generator")
lt_motor = Motor("LT-motor")
distribution = PowerBus("power-distribution", num_in=1, num_out=2)
grid = PowerSink("grid")

lt_e1 = PowerConnection(lt_turbine, "power", lt_generator, "power_in", label="LT-e1")
lt_e2 = PowerConnection(lt_generator, "power_out", distribution, "power_in1", label="LT-e2")

lt_e3 = PowerConnection(distribution, "power_out1", lt_motor, "power_in", label="LT-e3")
lt_e4 = PowerConnection(lt_motor, "power_out", lt_pump, "power", label="LT-e4")

e5 = PowerConnection(distribution, "power_out2", grid, "power", label="e5")

nw.add_conns(lt_e1, lt_e2, lt_e3, lt_e4, e5)

### 8. Parameters

#### pump and preheater

In [ ]:
c7.set_attr(fluid={"R245fa": 1}, p=1.55, x=0)             # pump inlet: sat. liquid
                                                           # NOTE: mass flow rate is NOT fixed here

lt_pump.set_attr(eta_s=0.70)
c8.set_attr(p=6.75)                                       # pump outlet

lt_preheater.set_attr(pr1=1, pr2=1)                       # no pressure drop assumed
c9.set_attr(x=0)                                          # preheater outlet: sat. liquid

#### evaporator

In [ ]:
lt_evaporator.set_attr(pr1=1, ttd_l=5)   # 5 K minimum approach temperature

c10.set_attr(p=6.75, x=1)                # evaporator outlet: SATURATED VAPOR (no superheat),
                                         # at the SAME pressure as the preheater outlet
                                         # -> zero pressure drop across preheater + evaporator

#### turbine and condenser

In [ ]:
lt_turbine.set_attr(eta_s=0.8272)
c10a.set_attr(p=1.55)                                       # turbine outlet pressure = condenser pressure

lt_condenser.set_attr(pr2=1)
# c7_in (condenser outlet, before the cycle closer) needs no separate spec:
# the CycleCloser enforces p, h, fluid and mass flow equal to c7 (state 7)

#### geothermal water side

In [ ]:
gC.set_attr(fluid={"water": 1}, T=92.95, p=10, m=122.37)  # evaporator inlet (point C)
# gD (evaporator water outlet, point D) is NOT fixed anymore - it is now a RESULT,
# since the evaporator duty is set by the working-fluid side (x=1 target + 5 K pinch),
# not by a pre-assumed water-side temperature drop.
gD_ph.set_attr(m=54.48)                                   # split to LT preheater (Table 4)
# gD_ht (67.89 kg/s) goes on to feed the HT-ORC preheater train - not modelled here

#### air side

In [ ]:
lt_a1.set_attr(fluid={"air": 1}, T=8.67, p=1.013)            # ambient air
lt_a2.set_attr(T=18.67)                                      # air outlet temperature
# air mass flow rate is NOT set -> TESPy solves for it from the condenser energy balance

#### power components

In [ ]:
lt_motor.set_attr(eta=1)
lt_generator.set_attr(eta=1)

### 10. Solve

In [ ]:
nw.solve("design")

## Results

### Overview and key results

In [ ]:
nw.print_results()

In [ ]:
nw.results["Connection"]

In [ ]:
nw.results["PowerConnection"]

In [ ]:
nw.results["SectionedHeatExchanger"]

In [ ]:
W_pump = lt_pump.P.val
W_turbine = lt_turbine.P.val          # negative sign = power output
Q_preheater = lt_preheater.Q.val
Q_evaporator = lt_evaporator.Q.val
Q_condenser = lt_condenser.Q.val

W_net = -W_turbine - W_pump           # kW
Q_in = -(Q_preheater + Q_evaporator)  # kW
eta_th = W_net / Q_in

print("================ LT-ORC SUMMARY ================")
print(f"Pump power         : {W_pump:8.1f} kW")
print(f"Turbine power      : {-W_turbine:8.1f} kW")
print(f"Preheater duty     : {-Q_preheater:8.1f} kW")
print(f"Evaporator duty    : {-Q_evaporator:8.1f} kW")
print(f"Condenser duty     : {-Q_condenser:8.1f} kW")
print(f"Air mass flow rate : {lt_a1.m.val:8.1f} kg/s")
print(f"Net power output   : {W_net:8.1f} kW")
print(f"Thermal efficiency : {eta_th * 100:8.2f} %")

### T-s diagram (LT-ORC)

The plot below shows the LT-ORC on temperature-entropy axes: the R245fa saturation dome, the five
state points (7-10a, matching the flow-chart numbering), and the process paths between them.

In [ ]:
FLUID = "R245fa"

def conn_Ts(conn):
    # T [degC] and s [kJ/kg-K] for a tespy connection (R245fa)
    p_Pa = conn.p.val * 1e5
    h_Jkg = conn.h.val * 1e3
    T = PropsSI("T", "P", p_Pa, "H", h_Jkg, FLUID) - 273.15
    s = PropsSI("S", "P", p_Pa, "H", h_Jkg, FLUID) / 1e3
    return T, s

def isobar_Ts(p_bar, h1_kJkg, h2_kJkg, n=40):
    # T,s along a constant-pressure line between two enthalpies
    p_Pa = p_bar * 1e5
    hs = np.linspace(h1_kJkg, h2_kJkg, n) * 1e3
    T = PropsSI("T", "P", p_Pa, "H", hs, FLUID) - 273.15
    s = PropsSI("S", "P", p_Pa, "H", hs, FLUID) / 1e3
    return T, s

# saturation dome
T_crit = PropsSI("Tcrit", FLUID)
T_dome = np.linspace(280, T_crit - 0.3, 200)
sf = PropsSI("S", "T", T_dome, "Q", 0, FLUID) / 1e3
sg = PropsSI("S", "T", T_dome, "Q", 1, FLUID) / 1e3
T_dome_C = T_dome - 273.15

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(np.concatenate([sf, sg[::-1]]), np.concatenate([T_dome_C, T_dome_C[::-1]]),
        "k-", lw=1, label="sat. dome")

# state points
# 7 = pump inlet, 8 = pump outlet, 9 = preheater outlet / evaporator inlet,
# 10 = evaporator outlet / turbine inlet, 10a = turbine outlet
states = {"7": c7, "8": c8, "9": c9, "10": c10, "10a": c10a}
pts = {k: conn_Ts(v) for k, v in states.items()}

# 7 -> 8: pump (real, non-isentropic process, straight line between end states)
ax.plot([pts["7"][1], pts["8"][1]], [pts["7"][0], pts["8"][0]], "b-", lw=2, label="pump / turbine")
# 8 -> 9: preheater (isobar)
T89, s89 = isobar_Ts(c8.p.val, c8.h.val, c9.h.val)
ax.plot(s89, T89, color="tab:orange", lw=2, label="preheater")
# 9 -> 10: evaporator (isobar, pressure interpolated across preheater -> evaporator outlet)
hs = np.linspace(c9.h.val, c10.h.val, 40) * 1e3
ps = np.linspace(c9.p.val, c10.p.val, 40) * 1e5
T910 = PropsSI("T", "P", ps, "H", hs, FLUID) - 273.15
s910 = PropsSI("S", "P", ps, "H", hs, FLUID) / 1e3
ax.plot(s910, T910, "r-", lw=2, label="evaporator")
# 10 -> 10a: turbine (real, non-isentropic process)
ax.plot([pts["10"][1], pts["10a"][1]], [pts["10"][0], pts["10a"][0]], "b-", lw=2)
# 10a -> 7: condenser (isobar)
T10a7, s10a7 = isobar_Ts(c10a.p.val, c10a.h.val, c7.h.val)
ax.plot(s10a7, T10a7, "c-", lw=2, label="condenser")

for k, (T, s) in pts.items():
    ax.plot(s, T, "ko", ms=5)
    ax.annotate(k, (s, T), textcoords="offset points", xytext=(6, 4))

ax.set_xlabel("specific entropy s [kJ/kg-K]")
ax.set_ylabel("temperature T [degC]")
ax.set_title("LT-ORC T-s diagram (R245fa)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### Pinch point check and heat exchanger T-Q diagrams

These plots show temperature vs. cumulative heat duty for each heat exchanger,
hot and cold streams together, assuming counter-current flow. The vertical gap
between the two curves at any point is the local approach temperature and
should match the `td_pinch` values from the pinch-point check in the cell
below. The deviation is from not including the exact phase change point in the
plots.

In [ ]:
print("Working fluid mass flow rate (SOLVED):", round(c7.m.val, 2), "kg/s")
print()
print(f"{"Component":<15}{"td_pinch [K]":>12}")
for hx in [lt_preheater, lt_evaporator, lt_condenser]:
    flag = "  <-- violation!" if hx.td_pinch.val < 0 else ""
    print(
        f"{hx.label:<15}{hx.td_pinch.val:>12.2f}"
        f"{flag}"
    )

In [ ]:
def counter_current_profile(m_hot, fluid_hot, p_hot_in_bar, p_hot_out_bar, h_hot_in, h_hot_out,
                             m_cold, fluid_cold, p_cold_in_bar, p_cold_out_bar, h_cold_in, h_cold_out,
                             n=40):
    # Q, T_hot, T_cold along a counter-current heat exchanger, evaluated at n
    # equally spaced points of cumulative heat duty Q (starting at the cold-inlet /
    # hot-outlet end, i.e. the coldest end of the exchanger).
    Q_total = m_cold * (h_cold_out - h_cold_in)  # kW
    Q = np.linspace(0, Q_total, n)
    h_cold = h_cold_in + Q / m_cold
    h_hot = h_hot_out + Q / m_hot
    p_cold = np.linspace(p_cold_in_bar, p_cold_out_bar, n) * 1e5
    p_hot = np.linspace(p_hot_out_bar, p_hot_in_bar, n) * 1e5
    T_cold = PropsSI("T", "P", p_cold, "H", h_cold * 1e3, fluid_cold) - 273.15
    T_hot = PropsSI("T", "P", p_hot, "H", h_hot * 1e3, fluid_hot) - 273.15
    return Q, T_hot, T_cold


fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))

# Preheater: hot = geothermal water (D -> E), cold = R245fa (8 -> 9)
Q, Th, Tc = counter_current_profile(
    gD_ph.m.val, "water", gD_ph.p.val, gE.p.val, gD_ph.h.val, gE.h.val,
    c8.m.val, "R245fa", c8.p.val, c9.p.val, c8.h.val, c9.h.val)
axs[0].plot(Q, Th, "r-o", ms=3, label="geothermal water")
axs[0].plot(Q, Tc, "b-o", ms=3, label="R245fa")
axs[0].set_title(f"LT Preheater (min approach = {(Th - Tc).min():.2f} K)")

# Evaporator: hot = geothermal water (C -> D), cold = R245fa (9 -> 10)
Q, Th, Tc = counter_current_profile(
    gC.m.val, "water", gC.p.val, gD.p.val, gC.h.val, gD.h.val,
    c9.m.val, "R245fa", c9.p.val, c10.p.val, c9.h.val, c10.h.val)
axs[1].plot(Q, Th, "r-o", ms=3, label="geothermal water")
axs[1].plot(Q, Tc, "b-o", ms=3, label="R245fa")
axs[1].set_title(f"LT Evaporator (min approach = {(Th - Tc).min():.2f} K)")

# Condenser: hot = R245fa (10a -> 7, cooling down), cold = air (7 -> 8)
Q, Th, Tc = counter_current_profile(
    c10a.m.val, "R245fa", c10a.p.val, c7.p.val, c10a.h.val, c7.h.val,
    lt_a1.m.val, "air", lt_a1.p.val, lt_a2.p.val, lt_a1.h.val, lt_a2.h.val)
axs[2].plot(Q, Th, "r-o", ms=3, label="R245fa")
axs[2].plot(Q, Tc, "b-o", ms=3, label="air")
axs[2].set_title(f"LT Condenser (min approach = {(Th - Tc).min():.2f} K)")

for ax in axs:
    ax.set_xlabel("cumulative heat duty Q [kW]")
    ax.set_ylabel("temperature T [degC]")
    ax.grid(alpha=0.3)
    ax.legend()

fig.tight_layout()
plt.show()